# Import Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Load and Preprocess Dataset

In [2]:
data = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance."""

# Tokenization and Numerical Representation

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

# Create Input-Output Sequence Pairs

In [4]:
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Build LSTM Model

In [ ]:
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train the Model

In [ ]:
model.fit(X, y, epochs=200, verbose=1)

# Generate New Sequences

In [ ]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print(generate_text("machine learning", 5))
print(generate_text("sequence models", 5))

# Component II: Transformer Based Sequence Generation

# Tokenization

In [28]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
index_to_word = {index: word for word, index in tokenizer.word_index.items()}

sequences = []
for line in data.split('\n'):
    seq = tokenizer.texts_to_sequences([line])[0]
    sequences.append(seq)

max_len = max(len(seq) for seq in sequences)
sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Positional Encoding

In [32]:
def positional_encoding(position, d_model):
    angles = np.arange(position)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :]//2)) / np.float32(d_model))
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles, dtype=tf.float32)

pos_encoding = positional_encoding(max_len, 32)

# Transformer Model

In [33]:
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dropout

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout = Dropout(0.1)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# Build Transformer Model

In [34]:
embed_dim = 32

inputs = tf.keras.Input(shape=(max_len - 1,))
x = Embedding(total_words, embed_dim)(inputs)

x = x + pos_encoding[:max_len - 1]

transformer_block = TransformerBlock(embed_dim, 2, 128)
x = transformer_block(x)

x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = Dense(total_words, activation="softmax")(x)

transformer_model = tf.keras.Model(inputs=inputs, outputs=outputs)
transformer_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Prepare Data for Transformer

In [35]:
X_t = sequences[:, :-1]
y_t = sequences[:, -1]

# Train Transformer Model

In [ ]:
transformer_model.fit(X_t, y_t, epochs=200, verbose=1)

# Generate Sequences using Transformer

In [ ]:
def generate_transformer(seed_text, next_words=5):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_text])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='post')

        pred = np.argmax(transformer_model.predict(seq), axis=-1)[0]
        next_word = index_to_word.get(pred, None)

        if next_word is None:
            break

        seed_text += " " + next_word

    return seed_text
print(generate_transformer("deep learning"))
print(generate_transformer("sequence models"))

In [ ]:
# Save LSTM model
model.save("lstm_sequence_model.h5")

# Save Transformer model
transformer_model.save("transformer_sequence_model.h5")

In [39]:
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)